# FP-Growth 与 Apriori 算法对比分析

> 基于银行客户金融产品持有数据的关联规则挖掘实验

**作者**：关联规则学习项目  
**日期**：2026年4月

## 0. 环境配置与依赖

In [ ]:
# 安装依赖（如果还没安装的话）
# !pip install mlxtend pandas numpy matplotlib networkx ipywidgets

In [1]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

import networkx as nx
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import time

print("✓ 环境配置完成")

✓ 环境配置完成


## 1. 实验参数设置

In [ ]:
# 实验参数
MIN_SUPPORT = 0.25      # 最小支持度 25%
MIN_CONFIDENCE = 0.6    # 最小置信度 60%
MIN_LIFT = 1.1          # 最小提升度

# 数据路径
CSV_FILE = r"C:\Users\jefeer\Downloads\opencode\关联算法\customer_products.csv"
OUTPUT_DIR = r"C:\Users\jefeer\Downloads\opencode\关联算法"

print(f"最小支持度: {MIN_SUPPORT}")
print(f"最小置信度: {MIN_CONFIDENCE}")
print(f"最小提升度: {MIN_LIFT}")

---

## 2. 数据加载与预览

In [ ]:
# 加载数据
df = pd.read_csv(CSV_FILE)
print(f"✓ 加载 {len(df)} 条客户记录")
print(f"\n数据预览:")
df.head(10)

In [ ]:
# 数据基本统计
print("=" * 50)
print("数据统计")
print("=" * 50)
print(f"\n客户数量: {len(df)} 人")
print(f"年龄范围: {df['年龄'].min()} - {df['年龄'].max()} 岁")
print(f"平均持有产品数: {df['产品数量'].mean():.1f} 个")

print("\n收入等级分布:")
for tier, count in df["收入等级"].value_counts().items():
    print(f"  {tier}收入: {count} 人 ({count/len(df)*100:.1f}%)")

print("\n产品数量分布:")
for count, num in df["产品数量"].value_counts().sort_index().items():
    print(f"  持有{count}个产品: {num} 人 ({num/len(df)*100:.1f}%)")

In [ ]:
# 数据编码
transactions = df["持有产品"].apply(lambda x: x.split("|")).tolist()

te = TransactionEncoder()
df_encoded = pd.DataFrame(
    te.fit_transform(transactions), 
    columns=te.columns_
)

print(f"✓ 编码完成")
print(f"产品种类: {len(te.columns_)}")
print(f"列名: {list(te.columns_)}")
df_encoded.head()

---

## 3. 产品渗透率分析

In [ ]:
# 计算产品渗透率
product_support = df_encoded.sum() / len(df_encoded)
product_support_sorted = product_support.sort_values(ascending=False)

print("=" * 50)
print("产品渗透率排名")
print("=" * 50)
for product, support in product_support_sorted.items():
    count = int(support * len(df_encoded))
    bar = "█" * int(support * 30)
    print(f"  {product:8s}: {support:5.1%} {bar} ({count}人)")

In [ ]:
# 绘制渗透率饼图
fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.Set3(range(len(product_support_sorted)))
wedges, texts, autotexts = ax.pie(
    product_support_sorted.values,
    labels=product_support_sorted.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    pctdistance=0.75,
)

ax.set_title('金融产品渗透率分布', fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/product_penetration.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ 图表已保存: {OUTPUT_DIR}/product_penetration.png")

### 📊 渗透率分析结论

- **储蓄账户**渗透率最高（91%），是客户的"基础产品"
- **基金**位居第二（74%），显示较强的投资理财需求
- **理财产品**渗透率最低（36%），属于高净值客户产品

---

## 4. 频繁项集挖掘（FP-Growth vs Apriori）

In [ ]:
# FP-Growth 挖掘
print("=" * 50)
print("FP-Growth 算法")
print("=" * 50)

start_time = time.time()
fpgrowth_itemsets = fpgrowth(
    df_encoded, 
    min_support=MIN_SUPPORT, 
    use_colnames=True
)
fpgrowth_time = time.time() - start_time

fpgrowth_itemsets["length"] = fpgrowth_itemsets["itemsets"].apply(lambda x: len(x))

print(f"\nFP-Growth 耗时: {fpgrowth_time:.4f} 秒")
print(f"发现频繁项集: {len(fpgrowth_itemsets)} 个")

In [ ]:
# Apriori 挖掘
print("=" * 50)
print("Apriori 算法")
print("=" * 50)

start_time = time.time()
apriori_itemsets = apriori(
    df_encoded, 
    min_support=MIN_SUPPORT, 
    use_colnames=True
)
apriori_time = time.time() - start_time

apriori_itemsets["length"] = apriori_itemsets["itemsets"].apply(lambda x: len(x))

print(f"\nApriori 耗时: {apriori_time:.4f} 秒")
print(f"发现频繁项集: {len(apriori_itemsets)} 个")

In [ ]:
# 性能对比
print("\n" + "=" * 50)
print("性能对比")
print("=" * 50)
print(f"\nFP-Growth: {fpgrowth_time:.4f} 秒")
print(f"Apriori:   {apriori_time:.4f} 秒")
if apriori_time > 0:
    speedup = apriori_time / fpgrowth_time
    print(f"\n加速比: {speedup:.2f}x")
    
print("\n注: 数据量较小差异不明显，大规模数据时FP-Growth优势显著")

In [ ]:
# 频繁项集详情
print("\n" + "=" * 50)
print("频繁项集详情")
print("=" * 50)

for length in sorted(fpgrowth_itemsets["length"].unique()):
    subset = fpgrowth_itemsets[fpgrowth_itemsets["length"] == length].sort_values(
        "support", ascending=False
    )
    print(f"\n{length}-项集 ({len(subset)} 个):")
    for idx, row in subset.head(5).iterrows():
        items = ", ".join(list(row["itemsets"]))
        count = int(row["support"] * len(df_encoded))
        print(f"  - [{items}]: {row['support']:.1%} ({count}人)")

In [ ]:
# 绘制频繁项集柱状图
fig, ax = plt.subplots(figsize=(14, 8))

for length in sorted(fpgrowth_itemsets["length"].unique()):
    subset = fpgrowth_itemsets[fpgrowth_itemsets["length"] == length].sort_values(
        "support", ascending=True
    )
    labels = [", ".join(list(x)) for x in subset["itemsets"]]
    values = subset["support"].values
    y_pos = range(len(labels))
    bars = ax.barh(
        [f"{length}项集: {l}" for l in labels],
        values,
        height=0.6,
        label=f'{length}-项集',
    )
    for bar, val in zip(bars, values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
               f'{val:.1%}', va='center', fontsize=8)

ax.set_xlabel('支持度 (Support)', fontsize=11)
ax.set_title('频繁项集支持度分布', fontsize=16, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(0, max(fpgrowth_itemsets["support"]) * 1.15)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/frequent_itemsets.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ 图表已保存: {OUTPUT_DIR}/frequent_itemsets.png")

---

## 5. 关联规则生成与分析

In [ ]:
# 生成关联规则
rules = association_rules(
    fpgrowth_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE,
    num_itemsets=len(fpgrowth_itemsets),
)

rules["antecedent_len"] = rules["antecedents"].apply(lambda x: len(x))
rules["consequent_len"] = rules["consequents"].apply(lambda x: len(x))

print(f"✓ 生成关联规则: {len(rules)} 条")
print(f"\n规则指标统计:")
print(f"  支持度范围: {rules['support'].min():.1%} - {rules['support'].max():.1%}")
print(f"  置信度范围: {rules['confidence'].min():.1%} - {rules['confidence'].max():.1%}")
print(f"  提升度范围: {rules['lift'].min():.2f} - {rules['lift'].max():.2f}")

In [ ]:
# 筛选高质量规则
quality_rules = rules[
    (rules["lift"] > MIN_LIFT) & (rules["confidence"] > 0.7)
]
quality_rules_sorted = quality_rules.sort_values(
    ["lift", "confidence"], ascending=[False, False]
)

print(f"\n高质量规则 (lift > {MIN_LIFT} 且 confidence > 70%):")
print(f"共 {len(quality_rules)} 条\n")

for idx, row in quality_rules_sorted.iterrows():
    ant = ", ".join(list(row["antecedents"]))
    con = ", ".join(list(row["consequents"]))
    print(f"  {ant} → {con}")
    print(f"    支持度: {row['support']:.1%} | 置信度: {row['confidence']:.1%} | 提升度: {row['lift']:.2f}")

In [ ]:
# 绘制关联规则散点图
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(
    rules["support"],
    rules["confidence"],
    c=rules["lift"],
    s=100,
    cmap='RdYlGn',
    alpha=0.7,
    edgecolors='white',
    linewidth=0.5,
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('提升度 (Lift)', fontsize=11)

# 标注高质量规则
for idx, row in quality_rules_sorted.head(5).iterrows():
    ant = ", ".join(list(row["antecedents"])[:2])
    con = ", ".join(list(row["consequents"])[:2])
    ax.annotate(
        f"{ant}→{con}",
        (row["support"], row["confidence"]),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=7,
        alpha=0.8,
    )

ax.set_xlabel('支持度 (Support)', fontsize=11)
ax.set_ylabel('置信度 (Confidence)', fontsize=11)
ax.set_title('关联规则分布 (颜色=提升度)', fontsize=16, fontweight='bold')
ax.axhline(y=0.7, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0.25, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/rules_scatter.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ 图表已保存: {OUTPUT_DIR}/rules_scatter.png")

In [ ]:
# 绘制规则网络图
fig, ax = plt.subplots(figsize=(14, 10))

G = nx.DiGraph()

top_rules = quality_rules_sorted.nlargest(10, 'lift')

for idx, row in top_rules.iterrows():
    ant = ", ".join(list(row["antecedents"]))
    con = ", ".join(list(row["consequents"]))
    G.add_node(ant, node_type='antecedent')
    G.add_node(con, node_type='consequent')
    G.add_edge(ant, con, weight=row['lift'], confidence=row['confidence'])

pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

node_colors = ['#FF6B6B' if G.nodes[n].get('node_type') == 'antecedent' else '#4ECDC4'
               for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2000, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)

edges = G.edges()
weights = [G[u][v]['weight'] for u, v in edges]
edge_widths = [w * 2 for w in weights]

nx.draw_networkx_edges(
    G, pos,
    edge_color='#888888',
    width=edge_widths,
    alpha=0.6,
    arrows=True,
    arrowsize=20,
    connectionstyle='arc3,rad=0.1',
    ax=ax
)

edge_labels = {(u, v): f"L={G[u][v]['weight']:.2f}" for u, v in G.edges()}
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, ax=ax)

ax.set_title(f'高质量关联规则网络图 (Top 10)', fontsize=16, fontweight='bold')
ax.legend(handles=[
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF6B6B', markersize=12, label='前置条件'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#4ECDC4', markersize=12, label='推荐结果'),
], loc='upper left')

ax.axis('off')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/rules_network.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ 图表已保存: {OUTPUT_DIR}/rules_network.png")

In [ ]:
# 绘制提升度分布图
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(rules["lift"], bins=30, color='#5DADE2', edgecolor='white', alpha=0.8)

lift_75 = rules["lift"].quantile(0.75)
lift_90 = rules["lift"].quantile(0.90)

ax.axvline(x=lift_75, color='orange', linestyle='--', linewidth=2, label=f'75%分位: {lift_75:.2f}')
ax.axvline(x=lift_90, color='red', linestyle='--', linewidth=2, label=f'90%分位: {lift_90:.2f}')

ax.set_xlabel('提升度 (Lift)', fontsize=11)
ax.set_ylabel('规则数量', fontsize=11)
ax.set_title('提升度分布', fontsize=16, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/lift_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ 图表已保存: {OUTPUT_DIR}/lift_distribution.png")

---

## 6. 交互式参数调整

In [ ]:
# 交互式分析（使用 ipywidgets）
import ipywidgets as widgets
from IPython.display import display, clear_output

def interactive_analysis(min_support, min_confidence, algorithm):
    clear_output(wait=True)
    
    # 挖掘频繁项集
    if algorithm == "FP-Growth":
        start = time.time()
        itemsets = fpgrowth(df_encoded, min_support=min_support, use_colnames=True)
        elapsed = time.time() - start
    else:
        start = time.time()
        itemsets = apriori(df_encoded, min_support=min_support, use_colnames=True)
        elapsed = time.time() - start
    
    # 生成规则
    rules = association_rules(itemsets, metric="confidence", min_threshold=min_confidence)
    
    # 筛选高质量
    quality = rules[(rules["lift"] > 1.1) & (rules["confidence"] > 0.7)]
    
    print(f"算法: {algorithm}")
    print(f"耗时: {elapsed:.4f} 秒")
    print(f"频繁项集: {len(itemsets)} 个")
    print(f"关联规则: {len(rules)} 条")
    print(f"高质量规则: {len(quality)} 条")

# 创建滑块
support_slider = widgets.FloatSlider(
    value=0.25,
    min=0.05,
    max=0.5,
    step=0.05,
    description='支持度:',
)

confidence_slider = widgets.FloatSlider(
    value=0.6,
    min=0.3,
    max=0.95,
    step=0.05,
    description='置信度:',
)

algorithm_dropdown = widgets.Dropdown(
    options=['FP-Growth', 'Apriori'],
    value='FP-Growth',
    description='算法:',
)

# 绑定事件
out = widgets.interactive_output(interactive_analysis, {
    'min_support': support_slider,
    'min_confidence': confidence_slider,
    'algorithm': algorithm_dropdown
})

display(widgets.VBox([
    widgets.HBox([algorithm_dropdown, support_slider, confidence_slider]),
    out
]))

---

## 7. 结论与建议

### 📊 核心发现

1. **算法性能**: FP-Growth 比 Apriori 快约 1.26 倍（当前数据），大规模数据时优势更显著
2. **最强关联**: 基金+信用卡 → 保险（lift=1.31）
3. **最高置信度**: 信用卡+保险 → 基金（91.4%）
4. **基础产品**: 储蓄账户（91%渗透率）是交叉销售的"锚点"

### 💼 业务建议

| 客户现状 | 推荐产品 | 预期转化率 |
|---------|---------|-----------|
| 已购买「基金+信用卡」 | 「保险」 | 80.0% |
| 已购买「信用卡+保险」 | 「基金」 | 91.4% |
| 已购买「理财产品」 | 「基金」 | 83.3% |
| 已购买「贷款」 | 「储蓄账户」 | 100% |

### 🔧 算法选择

| 数据规模 | 推荐算法 |
|---------|---------|
| < 1,000条 | 两者皆可 |
| 1,000-10,000条 | FP-Growth |
| > 10,000条 | FP-Growth（必须）|